# X5_Q4 — Parámetros como funciones dinámicas (calibración de diseño)

Primer notebook de la **Fase 2 de X5_alt_v2** (ver
[`docs/plans/plan_x5_alt_v2.md`](../docs/plans/plan_x5_alt_v2.md), que continúa
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10).
TODOs en `docs/tracking/todos.md`, sección X5_alt_v2 — **lista estrictamente
secuencial**: cada parámetro se trabaja en 3 pasos consecutivos
(*implementar* → *ejecutar* → *revisar visuales y validar*), no saltar al
siguiente parámetro sin cerrar los 3 del anterior.

Consume la evidencia ya cerrada de `X5_Q1.ipynb`/`X5_Q2.ipynb`/`X5_Q3.ipynb`
(`exp_Q1/{valor}_tabla_estado_v2.csv`, `exp_Q2/{valor}_q2_5_export_metricas.json`,
`exp_Q3/{valor}_q3_4_tabla_final.csv`) para calibrar, por parámetro, una sigmoide
acotada `P̂(s) = piso + (techo-piso)/(1+exp(-pendiente*(s-punto_medio)))`
contra un **target sintético de diseño** (percentil causal del índice
compuesto de 1-2 variables explicativas) — no un target económico de
backtesting. Esa validación económica (parámetro dinámico vs. estático real)
queda para `X5_Q5.ipynb`, fuera de alcance de este notebook.

**No implementa reglas de trading ni código de producción** — mismo criterio
de alcance que `Q3-3` aplicó a `X5_Q3.ipynb`.

Gate de arranque (`[Q4-1]`): requiere `Q3-EXEC` ya satisfecho — ver
`docs/tracking/done.md`.

Mismo input por lista `valores` que `X5_Q1`/`X5_Q2`/`X5_Q3.ipynb`.

**Estándar de documentación de esta fase (decisión 2026-10-17, ver
`docs/context/decisiones.md`)**: `Q1`/`Q2`/`Q3` son infraestructura de
investigación — Mauricio no necesita revisar cada celda. `Q4` (y `Q5`
después) son el notebook donde Mauricio sí se involucra directamente para
entender, cuestionar y decidir — por eso, a partir de la Sección 4, cada
parámetro se presenta con la secuencia Pregunta → Motivación → Metodología →
Fundamento matemático → Ejecución → Resultados → Interpretación →
Conclusión → Implicancias. El fundamento matemático y la metodología se
explican en detalle la primera vez que aparecen (Sección 4, `N`) y se
referencian sin repetir en las secciones siguientes (`A`, `factor_B`, `K`,
`N_EXP`) cuando el mecanismo es el mismo. Mismo estándar aplica, en
adelante, a `X5_Q5.ipynb` cuando se construya.

In [ ]:
import sys
from pathlib import Path

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

valores = list(cfg.VALORES)  # default = todos los activos de config.py; recortar a mano acá para un análisis puntual con menos

## 0. Preparación (Q4-1)

Imports y configuración común. Igual patrón que `X5_Q1`/`X5_Q2`/`X5_Q3.ipynb`:
`config.py` para rutas de `resources/x5_alt/`, opciones de despliegue de
`pandas`.

In [ ]:
import json

import pandas as pd

from x5_alt_funciones import calibrar_parametro
from x5_alt_export import carpeta_exp, limpiar_exp, exportar_zip  # noqa: E402

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

CARPETA_EXP = limpiar_exp('X5_Q4')  # limpia resources/x5_alt/exp_Q4 al inicio de cada corrida

print(f"Activos: {valores}")

## 1. Carga de outputs de Q1/Q2/Q3 (Q4-1)

Lee, por cada activo de `valores`, `{valor}_tabla_estado_v2.csv` (`X5_Q1.ipynb`),
`{valor}_q2_5_export_metricas.json` (`X5_Q2.ipynb`) y `{valor}_q3_4_tabla_final.csv`
(`X5_Q3.ipynb`) — ninguno se recalcula acá. Si falta alguno, corrió incompleto
el notebook correspondiente.

In [ ]:
def _cargar_tabla_estado(valor):
    path = carpeta_exp('X5_Q1') / f'{valor}_tabla_estado_v2.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q1.ipynb primero para generarla.")
    return pd.read_csv(path, parse_dates=['DateTime'])


def _cargar_export_metricas(valor):
    path = carpeta_exp('X5_Q2') / f'{valor}_q2_5_export_metricas.json'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q2.ipynb primero para generarlo.")
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def _cargar_tabla_final_q3(valor):
    path = carpeta_exp('X5_Q3') / f'{valor}_q3_4_tabla_final.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q3.ipynb primero para generarla.")
    return pd.read_csv(path)


tablas = {valor: _cargar_tabla_estado(valor) for valor in valores}
export_metricas = {valor: _cargar_export_metricas(valor) for valor in valores}
tablas_finales_q3 = {valor: _cargar_tabla_final_q3(valor) for valor in valores}

for valor in valores:
    tabla = tablas[valor]
    print(f"{valor}: tabla_estado {tabla.shape[0]} filas x {tabla.shape[1]} columnas — "
          f"{tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

## 2. Alcance de este notebook (Q4-2)

**`X5_Q4.ipynb` no implementa reglas de trading ni resultado económico.**
Calibra, por parámetro, una función de diseño `P̂(s)` acotada entre un piso y
un techo configurados (no necesariamente el rango completo de
`X5_PARAM_RANGES` — ver `plan_x5_alt_v2.md` sección 5), ajustada contra un
**target sintético por percentil causal** — no contra un óptimo económico
real (eso es `X5_Q5.ipynb`).

Quedan explícitamente fuera de este notebook:

- Cualquier wiring a `X0_data_supports.py`/`X1_trading.py`/`config.py` de
  producción.
- Separar `A` en `A_OE`/`A_TRAILING` — decisión pendiente de
  `x5_contexto_conversacion.md` §21; `A` se calibra único por ahora.
- `LAMBDA` — fuera del alcance comprometido de esta fase (decisión explícita
  de Mauricio). Queda como extensión opcional al final del notebook
  (Sección 11, `[Q4-L1]`-`[Q4-L4]`), condicional a resolver antes su
  bloqueador de rango (heredado de `Q3-6`).
- Validar si el parámetro dinámico mejora el resultado económico vs. el
  estático actual — eso es `X5_Q5.ipynb`.

Un buen ajuste acá (R² alto contra el target sintético) **no** es evidencia
de utilidad económica — es solo evidencia de que la sigmoide aproxima bien
el mapeo por percentiles que se le pidió aproximar.

## 3. Utilidades compartidas (Q4-2, Q4-3)

`scripts/x5_alt_funciones.py` — sigmoide acotada, índice compuesto (1-2
variables z-scoreadas), percentil causal sin leakage (`[Q4-3]`) y
`calibrar_parametro` (ajuste por `curve_fit` contra el target sintético; con
2 variables explora además el peso relativo por grid search de R², ver
docstring del módulo). Una sola implementación — las secciones 4-9 la
reutilizan, nunca reimplementan la fórmula. Importado ya en la Sección 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from x5_alt_funciones import percentil_causal

VARIABLE_INSPECCION_Q4_3 = 'x3_drawdown_20'  # inspección puntual de Q4-3; no es la variable final de ningún parámetro

for valor in valores:
    tabla = tablas[valor]
    q_t = percentil_causal(tabla[VARIABLE_INSPECCION_Q4_3].to_numpy())

    fig, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(tabla['DateTime'], tabla['Close'], color='#1f77b4', linewidth=0.8)
    ax1.set_ylabel('Close', color='#1f77b4')
    ax1.tick_params(axis='y', labelcolor='#1f77b4')

    ax2 = ax1.twinx()
    ax2.plot(tabla['DateTime'], q_t, color='#d62728', linewidth=0.8, alpha=0.85)
    ax2.set_ylabel(f'percentil_causal({VARIABLE_INSPECCION_Q4_3})', color='#d62728')
    ax2.tick_params(axis='y', labelcolor='#d62728')
    ax2.set_ylim(0, 1)

    ax1.set_xlabel('DateTime')
    ax1.set_title(f"{valor} — Close (izq.) vs. percentil_causal({VARIABLE_INSPECCION_Q4_3}) (der.)")
    plt.tight_layout()
    plt.show()

    print(f"{valor}: percentil_causal({VARIABLE_INSPECCION_Q4_3}) — "
          f"media {q_t.mean():.3f}, p05 {np.quantile(q_t, 0.05):.3f}, "
          f"p50 {np.quantile(q_t, 0.5):.3f}, p95 {np.quantile(q_t, 0.95):.3f}")


## 4. N = f(drawdown, volatilidad) — ¿cuántos soportes conviene mantener activos según el estado del activo?

**Pasos**: `[Q4-4]` implementar → `[Q4-5]` ejecutar `X5_Q4.ipynb` completo →
`[Q4-6]` revisar visuales y validar con Mauricio antes de pasar a `A`.

### 4.0. Qué estamos investigando y por qué

`N` (`n_sizes_ejecucion` en `config.py`) es la cantidad de soportes que X1
mantiene activos para un activo. Más `N` reparte el capital en más niveles
(entradas más finas, pero cada una más chica); menos `N` concentra el
capital en menos niveles (entradas más grandes, pero menos cobertura del
rango de precio). Hoy `N` es un número fijo por activo en `config.py` — la
pregunta de esta sección es si conviene que `N` **respire** con el estado
del activo en vez de quedarse fijo.

La hipótesis de partida (`x5_contexto_conversacion.md` §6, confirmada en
`X5_alternativo.md` §10.5) es puntual:

> drawdown sano (el precio está cerca de su máximo reciente) **y**
> volatilidad baja → conviene un `N` alto (fragmentar más, el activo está
> "tranquilo"). drawdown profundo **y** volatilidad alta → conviene un `N`
> bajo (concentrar menos, el activo está en una zona de mayor riesgo).

Esta sección **no** prueba si esa hipótesis mejora el resultado económico de
X1 — eso es `X5_Q5.ipynb`. Responde una pregunta más chica y previa: ¿existe
una función matemática simple, acotada entre los límites ya validados de
`N`, que traduzca el estado del activo (drawdown + volatilidad) en un valor
de `N` coherente con esa hipótesis? Si la respuesta es no — la curva no
ajusta bien, o el signo sale al revés de lo esperado — no tiene sentido
pasar a `Q5` a probarla económicamente.

La sección sigue este camino: índice de estado del activo → ajuste de la
sigmoide contra un target de diseño → comparación visual con el valor
estático actual → diagnóstico numérico → exportación. El mecanismo completo
se explica en detalle acá porque es el mismo que reutilizan, sin cambios de
diseño, las Secciones 5-8 (`A`, `factor_B`, `K`, `N_EXP`).

### 4.1. Variables elegidas y por qué

- **`x3_drawdown_20`**: de las 3 variables de drawdown que `Q3` evaluó para
  `N` (`x3_drawdown_20`, `x3_drawdown_50`, `drawdown_7d`), es la que mostró
  mejor evidencia descriptiva contra retornos (Pearson 0.28, similitud
  coseno 0.21 — `plan_x5_alt_v2.md` §2).
- **`x3_atr_pct` con el signo invertido (`-x3_atr_pct`)**: `Q3` no había
  testeado volatilidad junto a `N` — se agrega acá para completar la
  hipótesis "drawdown + volatilidad" (`X5_alternativo.md` §10.5). Se
  invierte el signo porque, en magnitud cruda, `drawdown_20` alto es un
  estado sano del activo, pero `atr_pct` alto es lo contrario (más volátil
  = peor para esta hipótesis) — las dos variables deben apuntar en la misma
  dirección antes de combinarse (ver 4.2), y la convención elegida es que el
  índice combinado `s_t` alto signifique siempre "estado sano → `N` alto".

Rango de `N`: `(50,200)` para BTCUSD/ETHUSD, `(40,180)` para el resto
(`X5_PARAM_RANGES['n_sizes_ejecucion']`, definido localmente en el notebook
por grupo de activo porque ese diccionario de `config.py` todavía no tiene
entrada para activos agregados después de armarlo, ej. `MSFT`).

### 4.2. Metodología: de dos variables crudas a un único índice de estado `s_t`

No se puede ajustar una curva con dos variables de entrada directamente —
`drawdown_20` se mueve en una escala (fracción del máximo histórico) y
`atr_pct` en otra (porcentaje del precio). El primer paso es ponerlas en una
escala comparable y combinarlas en un solo número por barra, `s_t`.

**Paso 1 — normalizar cada variable (z-score)**:

$$z_{i,t} = \frac{x_{i,t} - \mu_i}{\sigma_i}$$

`μ_i`/`σ_i` son la media y la desviación estándar de la variable sobre todo
el histórico usado para calibrar (se calculan una sola vez, igual que la
normalización de un modelo ya entrenado — no se recalculan barra a barra).
El z-score mide cuántas desviaciones estándar está un valor por encima o
por debajo de su propio promedio histórico: `z=0` es "en el promedio",
`z=+2` es "inusualmente alto para este activo", `z=-2` "inusualmente bajo".
En la misma escala, `drawdown_20` y `-atr_pct` ya se pueden sumar sin que
una domine a la otra solo por tener números más grandes en magnitud cruda.

**Paso 2 — combinar en un índice ponderado**:

$$s_t = w_1 \, z_{1,t} + w_2 \, z_{2,t}, \qquad w_2 = 1 - w_1$$

Con 2 variables (caso de `N`), `w_1` es el único peso libre y se explora por
grid search en el ajuste (ver 4.4) — no se fija a mano. `s_t` alto significa
"estado combinado sano" (por construcción, tras invertir `atr_pct`); `s_t`
bajo significa "estado combinado adverso".

**Paso 3 — suavizar el índice antes de seguir (`SPAN_SUAVIZADO_N = 6`,
decisión de Mauricio 2026-10-17)**: la primera corrida de esta sección
mostró `N̂_t` saltando hasta ±70 en una hora incluso sin novedad real en el
activo — porque `x3_drawdown_20` resetea seguido contra su máximo rolling de
20 barras, generando ruido bar-a-bar que no refleja un cambio real y
sostenido del estado del activo. La regla para los 5 parámetros de esta fase
es que su movimiento debe ser suave salvo que el activo cambie de forma
sostenida (ej. una caída fuerte de precio en un día) — eso sí debe mover el
parámetro con fuerza. Se eligió un EMA (promedio móvil exponencial) causal
corto sobre `s_t`, no sobre `N̂_t` ya calculado:

$$s^{ema}_t = \alpha \, s_t + (1-\alpha) \, s^{ema}_{t-1}$$

con `α` derivado de `span=6` barras (`pandas.ewm(span=6, adjust=False)`).
"Causal" quiere decir que `s^{ema}_t` solo usa información hasta `t` — nunca
una barra futura (Principio 11 del documento rector,
`x5_contexto_conversacion.md`). Un EMA corto deja pasar un drawdown real
sostenido en pocas horas (la señal se mantiene extrema barra tras barra y el
promedio la va alcanzando), pero filtra el ruido de una sola barra que
resetea y vuelve. Validado antes de fijar `span=6`: comparación visual
contra `span=12`/`24` sobre BTCUSD (`plan_x5_alt_v2.md` §3.1) y contra el
crash más fuerte de la muestra (BTCUSD, -19% en 24h, jun-2022) — el EMA
sigue llegando a los extremos del rango en pocas horas ahí, no amortigua un
movimiento real.

**Paso 4 — construir un target de diseño sin backtesting (percentil
causal)**: todavía no existe un "valor óptimo histórico" de `N` por barra
contra el cual ajustar una curva — eso requeriría el backtesting condicional
que es, precisamente, el trabajo de `X5_Q5.ipynb`. En su lugar se usa un
target sintético que solo codifica la dirección ya hipotetizada y los
límites ya configurados:

$$q_t = \text{percentil\_causal}(s^{ema}_t) \in (0,1]$$

el percentil histórico de `s^{ema}_t`, calculado usando solo los datos hasta
`t` (expanding, nunca información futura — mismo Principio 11). Si
`s^{ema}_t` es el valor más alto visto hasta ese momento, `q_t≈1`; si es el
más bajo, `q_t≈0`.

$$y^*_t = \text{piso} + (\text{techo}-\text{piso}) \cdot q_t$$

es decir: si el estado combinado es el peor visto hasta ahora, el target es
el piso del rango de `N`; si es el mejor visto hasta ahora, el target es el
techo. Es un target de **diseño**, no de resultado económico — por eso un
buen ajuste acá demuestra que la curva aproxima bien ese mapeo por
percentiles, no que el `N` resultante genera mejor retorno. Esa distinción
es la razón por la que esta fase se llama "calibración de diseño" y no
"validación".

In [ ]:
RANGO_N = {valor: (50, 200) if valor in ('BTCUSD', 'ETHUSD') else (40, 180) for valor in valores}
VARIABLES_N = ['x3_drawdown_20', 'x3_atr_pct']
SPAN_SUAVIZADO_N = 6  # EMA causal en barras H1 — ver celda markdown de la Sección 4

resultados_n = {}

for valor in valores:
    tabla = tablas[valor].dropna(subset=VARIABLES_N).reset_index(drop=True)
    piso, techo = RANGO_N[valor]

    drawdown_20 = tabla['x3_drawdown_20'].to_numpy()
    atr_pct_invertido = -tabla['x3_atr_pct'].to_numpy()  # signo invertido — ver celda markdown de la Sección 4

    ajuste = calibrar_parametro([drawdown_20, atr_pct_invertido], piso, techo, span_suavizado=SPAN_SUAVIZADO_N)
    resultados_n[valor] = {'tabla': tabla, 'ajuste': ajuste}

    mu_dd, sigma_dd = ajuste['normalizacion'][0]['mu'], ajuste['normalizacion'][0]['sigma']
    mu_atr, sigma_atr = ajuste['normalizacion'][1]['mu'], ajuste['normalizacion'][1]['sigma']
    z_drawdown = (drawdown_20 - mu_dd) / sigma_dd
    z_atr_invertido = (atr_pct_invertido - mu_atr) / sigma_atr

    fig, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(tabla['DateTime'], tabla['Close'], color='#1f77b4', linewidth=0.8)
    ax1.set_ylabel('Close', color='#1f77b4')
    ax1.tick_params(axis='y', labelcolor='#1f77b4')

    ax2 = ax1.twinx()
    ax2.plot(tabla['DateTime'], z_drawdown, color='#2ca02c', linewidth=0.6, alpha=0.7, label='z(x3_drawdown_20) (sin suavizar)')
    ax2.plot(tabla['DateTime'], z_atr_invertido, color='#ff7f0e', linewidth=0.6, alpha=0.7, label='z(-x3_atr_pct) (sin suavizar)')
    ax2.plot(tabla['DateTime'], ajuste['s'], color='#d62728', linewidth=0.9, label=f's_t (índice combinado, EMA span={SPAN_SUAVIZADO_N})')
    ax2.set_ylabel('z-score', color='#d62728')
    ax2.tick_params(axis='y', labelcolor='#d62728')
    ax2.legend(fontsize=8, loc='upper left')

    ax1.set_xlabel('DateTime')
    ax1.set_title(f"{valor} — N: índice compuesto s_t (pesos {[round(w, 2) for w in ajuste['pesos']]})")
    plt.tight_layout()
    plt.show()


### 4.3. Cómo leer el gráfico anterior

El eje izquierdo (azul) es el precio de cierre, para ubicar visualmente
episodios conocidos del activo (caídas, rallies, lateralizaciones). El eje
derecho muestra tres series en z-score:

- `z(x3_drawdown_20)` y `z(-x3_atr_pct)` **sin suavizar** — el insumo crudo,
  ya normalizado pero todavía con el ruido bar-a-bar.
- `s_t` **ya suavizado** (EMA span=6) — el índice que realmente alimenta el
  ajuste de la sigmoide.

Lo que conviene revisar visualmente acá, antes de mirar el ajuste: que
`s_t` caiga claramente cuando el precio está en una caída marcada (coherente
con "drawdown profundo + volatilidad alta → estado adverso"), que se
recupere cuando el precio se recupera, y que la versión suavizada (roja) ya
no tenga los saltos bruscos bar-a-bar de las series crudas (verde/naranja).
Si `s_t` no se mueve en la dirección esperada durante un episodio conocido
(ej. no cae durante una caída fuerte de precio), es señal de revisar la
hipótesis o las variables antes de seguir al ajuste.

### 4.4. Fundamento matemático: la sigmoide acotada y cómo se ajusta

La forma elegida para los 5 parámetros de esta fase es siempre la misma
sigmoide (logística) acotada:

$$\hat P(s) = \text{piso} + \frac{\text{techo}-\text{piso}}{1+\exp(-\text{pendiente}\cdot(s-\text{punto\_medio}))}$$

Por qué esta forma y no, por ejemplo, una recta:

- **Queda acotada por construcción** entre `piso` y `techo` — nunca puede
  devolver un valor fuera del rango ya validado de `N` (`X5_PARAM_RANGES`),
  a diferencia de un ajuste lineal sin tope, que sí podría extrapolar fuera
  de rango con datos extremos.
- **Transición suave, sin quiebres** — evita que `N̂` salte de golpe si el
  estado del activo oscila justo alrededor de un límite de régimen.
- **Pocos números a ajustar e interpretables**: solo `punto_medio` y
  `pendiente` (más `w_1` si hay 2 variables) — deliberadamente pocos, para
  no sobreajustar con el historial disponible de cada activo.

**Cómo se ajustan `punto_medio`/`pendiente`**: por mínimos cuadrados
(`scipy.optimize.curve_fit`). La idea de mínimos cuadrados es simple: de
todas las curvas posibles de esta familia (variando `punto_medio` y
`pendiente`), se elige la que minimiza la suma de los errores al cuadrado
entre la curva y cada punto histórico `(s_t, y^*_t)`:

$$(\text{punto\_medio}^*, \text{pendiente}^*) = \arg\min \sum_t \big(y^*_t - \hat P(s_t)\big)^2$$

Elevar el error al cuadrado antes de sumarlo castiga más los errores grandes
que los chicos (un error de 20 pesa 4 veces más que uno de 10, no el doble)
y evita que errores positivos y negativos se cancelen entre sí.

**Cuando hay 2 variables (caso de `N`)**: el peso `w_1` no se puede ajustar
junto con `punto_medio`/`pendiente` en el mismo `curve_fit`, porque el
propio target `y^*_t` depende de `w_1` (cambiar el peso cambia `s_t`, que
cambia el percentil causal, que cambia el target completo). En vez de eso,
`calibrar_parametro` prueba una grilla de valores de `w_1` (de 0.1 a 0.9),
ajusta la sigmoide completa para cada uno, y se queda con el que da mejor
`R²` — un barrido de verosimilitud sobre `w_1`, no un ajuste conjunto de los
3 números a la vez.

In [ ]:
from x5_alt_funciones import sigmoide_acotada

for valor in valores:
    tabla = resultados_n[valor]['tabla']
    ajuste = resultados_n[valor]['ajuste']
    piso, techo = RANGO_N[valor]

    s_curva = np.linspace(ajuste['s'].min(), ajuste['s'].max(), 200)
    curva = sigmoide_acotada(s_curva, piso, techo, ajuste['punto_medio'], ajuste['pendiente'])

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(ajuste['s'], ajuste['y_objetivo'], s=4, alpha=0.15, color='#7f7f7f', label='(s_t, y*_t) histórico')
    ax.plot(s_curva, curva, color='#d62728', linewidth=2, label='P̂(s) ajustada')
    ax.axvline(ajuste['punto_medio'], color='#1f77b4', linestyle='--', linewidth=0.8, label='punto_medio')
    ax.set_xlabel('s_t (índice combinado)')
    ax.set_ylabel('N')
    ax.set_title(f"{valor} — N: curva ajustada sobre target sintético (R²={ajuste['r2']:.3f})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


### 4.5. Cómo leer el gráfico de la curva ajustada

Cada punto gris es una barra histórica: su posición horizontal es `s_t` (el
índice de estado de esa barra) y su posición vertical es `y^*_t` (el target
sintético que le corresponde según su percentil causal). La nube de puntos
grises **es**, por construcción, un mapeo por percentiles — no mide nada
económico, solo "qué tan bueno fue el estado combinado hasta esa barra,
comparado con el resto de la historia vista hasta entonces".

La curva roja es `P̂(s)` (la sigmoide de 4.4). Tres cosas para leer en ella:

- **`punto_medio`** (línea azul punteada): el valor de `s_t` donde la curva
  pasa exactamente por la mitad entre piso y techo — el "punto de
  indiferencia" del índice. A la izquierda de esa línea el estado se lee
  como predominantemente adverso; a la derecha, como predominantemente
  sano.
- **`pendiente`**: qué tan abrupta es la transición entre piso y techo. Una
  pendiente alta significa que `N` cambia rápido con pequeños cambios de
  `s_t` cerca del punto medio (la curva se parece más a un escalón); una
  pendiente baja significa una transición gradual (más parecida a una
  recta suave).
- **`R²`** (en el título): la fracción de la variación de `y^*_t` que la
  curva roja logra explicar. `R²=1` significaría que la sigmoide pasa
  exactamente por el centro de la nube de puntos en todo el rango; `R²=0`
  significaría que la curva no explica nada mejor que usar el promedio de
  `y^*_t` como predicción constante. Como el target ya es, por diseño, una
  transformación monótona de `s_t`, se espera un `R²` razonablemente alto —
  un `R²` bajo acá es señal de que la sigmoide no es buena familia funcional
  para este mapeo, o de que `s_t` tiene mucho ruido residual incluso después
  del suavizado.

Que la curva ajuste bien a esta nube de puntos **no** significa todavía que
el `N` resultante sea económicamente mejor que el estático actual — solo
significa que la fórmula matemática aproxima bien el mapeo por percentiles
que se le pidió aproximar. Esa es la limitación central de esta fase
(`plan_x5_alt_v2.md` §4, §11) y la razón de que `X5_Q5.ipynb` sea un gate
obligatorio antes de cualquier wiring a producción.

In [ ]:
for valor in valores:
    tabla = resultados_n[valor]['tabla']
    ajuste = resultados_n[valor]['ajuste']
    piso, techo = RANGO_N[valor]
    valor_estatico = cfg.n_sizes_ejecucion[valor]

    fig, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(tabla['DateTime'], tabla['Close'], color='#1f77b4', linewidth=0.8)
    ax1.set_ylabel('Close', color='#1f77b4')
    ax1.tick_params(axis='y', labelcolor='#1f77b4')

    ax2 = ax1.twinx()
    ax2.plot(tabla['DateTime'], ajuste['y_ajustado'], color='#d62728', linewidth=0.8, label='N̂_t (dinámico)')
    ax2.axhline(valor_estatico, color='#2ca02c', linestyle='--', linewidth=1, label=f'N estático ({valor_estatico})')
    ax2.set_ylabel('N', color='#d62728')
    ax2.tick_params(axis='y', labelcolor='#d62728')
    ax2.set_ylim(piso, techo)
    ax2.legend(fontsize=8, loc='upper left')

    ax1.set_xlabel('DateTime')
    ax1.set_title(f"{valor} — N̂_t dinámico vs. estático ({valor_estatico})")
    plt.tight_layout()
    plt.show()


### 4.6. Cómo leer el gráfico dinámico vs. estático

Esta es la pieza central de la validación visual que hace Mauricio en
`Q4-6`. La línea verde punteada es el `N` fijo que usa X1 hoy en producción
(`cfg.n_sizes_ejecucion[valor]`); la línea roja es `N̂_t`, el valor que
tomaría `N` en cada barra si ya estuviera calibrado y en vivo.

Preguntas concretas a responder mirando el gráfico, activo por activo:

1. **¿`N̂_t` baja durante episodios de caída de precio conocidos y sube
   durante recuperaciones?** Si la hipótesis se sostiene, la línea roja
   debería moverse de forma coherente con las caídas/recuperaciones
   visibles en el precio (eje izquierdo).
2. **¿El movimiento es suave, no errático?** Tras el suavizado de 4.2 no
   debería haber saltos grandes bar-a-bar fuera de episodios de cambio real
   y sostenido del activo.
3. **¿Dónde pasa más tiempo `N̂_t` respecto al estático?** Si pasa la mayor
   parte del tiempo muy por debajo o muy por encima del valor estático
   actual, vale la pena entender por qué — puede ser evidencia real de que
   el estático está mal calibrado, o puede ser un artefacto del punto medio
   elegido por el ajuste.

Ninguna de estas tres observaciones por sí sola valida ni invalida la
hipótesis — son la base empírica sobre la que Mauricio decide, en `Q4-6`, si
seguir a `A` con este mismo esquema o ajustar algo de `N` antes.

In [ ]:
for valor in valores:
    ajuste = resultados_n[valor]['ajuste']
    signo_coincide = ajuste['pendiente'] > 0
    print(f"{valor}: R²={ajuste['r2']:.3f}, elasticidad={ajuste['elasticidad_punto_medio']:.2f}, "
          f"pendiente={ajuste['pendiente']:.3f}, pesos={[round(w, 2) for w in ajuste['pesos']]} "
          f"— signo {'coincide' if signo_coincide else 'NO COINCIDE'} con la hipótesis Q3 "
          f"(drawdown sano + volatilidad baja → N alto)")
    if not signo_coincide:
        print(f"  ALERTA: revisar la hipótesis o el índice compuesto para {valor} antes de validar visualmente.")


### 4.7. Interpretación de los diagnósticos, conclusión e implicancias

**Qué mide cada número:**

- **`elasticidad_punto_medio`** = `pendiente * (techo-piso) / 4` — es la
  pendiente de `P̂(s)` evaluada exactamente en el punto medio (el punto más
  empinado de la sigmoide, por su forma). En términos prácticos: "cuánto
  cambia `N̂` por cada unidad que cambia `s_t`, justo en la zona donde `N̂`
  es más sensible al estado del activo". Una elasticidad alta significa que
  cerca del punto medio `N̂` reacciona con fuerza a pequeños cambios de
  estado; una elasticidad baja significa una reacción más tenue incluso ahí.
- **`pesos`** (`w_1`, `w_2`) — cuánto aporta cada variable al índice
  combinado. Si uno de los dos pesos queda cerca de 0, en la práctica esa
  variable no está aportando información nueva al ajuste (el `R²` sería
  casi igual usando solo la otra) — vale la pena revisarlo, no descartarlo
  de entrada.
- **Chequeo de signo** — la pendiente se espera positiva por construcción
  (el índice ya queda orientado "sano = alto" tras invertir `atr_pct`, ver
  4.1). Un signo negativo no sería un bug de cómputo, sería evidencia de
  que, para ese activo en particular, el ajuste encontró una relación
  contraria a la hipotetizada — motivo explícito para no avanzar a `Q4-6`
  sin revisar antes el índice o la hipótesis para ese activo puntual.

**Conclusión de esta sección** (a completar/confirmar por Mauricio en
`Q4-6` con los 7 activos ya ejecutados): si el signo coincide y el
comportamiento visual del punto 4.6 es coherente en la mayoría de los
activos, la hipótesis "drawdown sano + volatilidad baja → `N` alto" queda
**respaldada como diseño matemático válido** — no como mejora económica
confirmada, eso sigue pendiente de `Q5`.

**Implicancia para el resto de la fase**: el mecanismo completo (índice
compuesto → suavizado EMA → percentil causal → sigmoide por `curve_fit`) ya
quedó validado en código y, si `Q4-6` lo confirma visualmente, es el mismo
mecanismo que se reutiliza sin cambios de diseño para `A`, `factor_B`, `K` y
`N_EXP` (Secciones 5-8) — cambia la variable de entrada y el rango, no la
metodología.

### 4.8. Guardar la serie y los parámetros ajustados

Export a `{valor}_q4_n.csv`: la serie completa `s_t`/`y^*_t`/`N̂_t` barra a
barra, más los parámetros ajustados (`punto_medio`, `pendiente`, pesos,
`piso`/`techo`, `R²`, elasticidad) repetidos en cada fila — para que el
resumen consolidado de la Sección 9 y `X5_Q5.ipynb` puedan leer ambos niveles
(serie temporal y parámetros del combo) desde el mismo archivo.

In [ ]:
for valor in valores:
    tabla = resultados_n[valor]['tabla']
    ajuste = resultados_n[valor]['ajuste']
    piso, techo = RANGO_N[valor]

    export_n = pd.DataFrame({
        'DateTime': tabla['DateTime'],
        's_t': ajuste['s'],
        'y_objetivo': ajuste['y_objetivo'],
        'N_hat': ajuste['y_ajustado'],
    })
    export_n['punto_medio'] = ajuste['punto_medio']
    export_n['pendiente'] = ajuste['pendiente']
    export_n['peso_x3_drawdown_20'] = ajuste['pesos'][0]
    export_n['peso_x3_atr_pct_invertido'] = ajuste['pesos'][1]
    export_n['piso'] = piso
    export_n['techo'] = techo
    export_n['r2'] = ajuste['r2']
    export_n['elasticidad_punto_medio'] = ajuste['elasticidad_punto_medio']

    path = CARPETA_EXP / f'{valor}_q4_n.csv'
    export_n.to_csv(path, index=False)
    print(f"{valor}: {path.name} guardado — {export_n.shape[0]} filas")


## 5. A = f(volatilidad) — ¿cuánto debe alejarse el precio de un soporte antes de operar sobre él?

**Pasos**: `[Q4-7]` implementar → `[Q4-8]` ejecutar → `[Q4-9]` revisar
visuales y validar antes de pasar a `factor_B`.

### 5.0. Qué controla `A` en producción

`A` es un umbral monetario en USD que hoy cumple dos roles distintos en X1
(`x5_contexto_conversacion.md` §7):

1. **Creación de Órdenes en Espera**: un soporte recibe un buy limit cuando
   `(P0 - Pi) * L >= A` — un `A` alto exige que el soporte esté más alejado
   del precio actual antes de declarar la orden (menos órdenes, más
   separadas).
2. **Activación del primer trailing stop ganador**: mientras una posición
   no tiene stop loss, el primer stop se activa cuando la ganancia
   flotante `>= A` — un `A` alto retrasa esa primera protección.

**Complicación conocida, documentada pero no resuelta en esta fase**: estos
dos roles pueden empujar en direcciones distintas ante el mismo estado del
activo (un `A` alto filtra entradas nuevas, pero también retrasa proteger
una posición ganadora) — por eso `x5_contexto_conversacion.md` §21 deja
pendiente separar `A_OE` de `A_TRAILING`. Esta sección calibra un único `A`
para ambos roles; el resultado debe leerse con esa mezcla en mente, no como
si `A` tuviera un solo efecto limpio.

### 5.1. Hipótesis vigente

> volatilidad alta → `A` más alto (evitar órdenes demasiado cercanas cuando
> el activo se mueve con amplitud). volatilidad baja → `A` más bajo.

### 5.2. Variable elegida y por qué

*Pendiente de implementar.* Variable: `x3_atr_pct` (se descarta
`x3_bb_width` por redundante con `atr_pct`, Pearson 0.74, `Q2-12`). Rango:
`(2.0, 20.0)` USD (`X5_PARAM_RANGES['A']`), `k=1` (una sola variable, no hay
peso que ponderar).

### 5.3. Qué se reportará

Mismo mecanismo ya validado en la Sección 4 (índice de estado → suavizado
EMA → percentil causal → sigmoide por `curve_fit`), sin reexplicarlo acá —
con `k=1` no hay paso de ponderación de variables (4.2, Paso 2 se salta).
Mismos entregables: gráfico de la variable normalizada vs. precio, curva
ajustada sobre el target sintético, serie `Â_t` vs. el valor estático
actual, diagnóstico (R², elasticidad, chequeo de signo) y export a
`{valor}_q4_a.csv`.

## 6. factor_B = f(volatilidad, tendencia) — ¿qué tan ajustado debe ir el trailing stop?

**Pasos**: `[Q4-10]` implementar → `[Q4-11]` ejecutar → `[Q4-12]` revisar
visuales y validar antes de pasar a `K`.

### 6.0. Qué controla `factor_B` en producción

`B` ya no es un parámetro independiente — se redefinió como `B = factor_B *
A` (`x5_contexto_conversacion.md` §8, decisión confirmada §21). Una vez
activado el trailing stop, `B` es la distancia monetaria que se mantiene
entre el precio y el stop (`sl_nuevo = P0 - B/L`; el stop solo sube, nunca
retrocede). `factor_B` (adimensional) controla qué proporción de `A` se usa
como ese margen: factor bajo → trailing más ajustado (asegura ganancias más
rápido, pero más expuesto a cerrar por ruido); factor alto → trailing más
holgado (deja correr la posición, pero devuelve más ganancia si el precio
revierte).

### 6.1. Hipótesis vigente

> volatilidad alta → `factor_B` más alto (evitar que el ruido normal active
> el stop antes de tiempo). tendencia fuerte y persistente → `factor_B` más
> alto (dejar correr la posición). mercado lateral o muy ruidoso →
> `factor_B` bajo/intermedio — a confirmar con backtesting, un trailing muy
> ajustado puede cerrar por ruido y uno muy holgado puede devolver demasiada
> ganancia (`x5_contexto_conversacion.md` §8).

**Importante**: `factor_B` nunca debe optimizarse como si fuera
independiente de `A` — un `A` alto y un `factor_B` alto producen un `B`
absoluto mucho mayor. Esta sección calibra `factor_B` solo (no la
interacción con el `Â_t` de la Sección 5); la interacción real queda para la
validación económica de `Q5`.

### 6.2. Variables elegidas y por qué

*Pendiente de implementar.* Variables: `x3_atr_pct` (volatilidad) + `adx_14`
(tendencia — se descarta `regimen_tendencia` por ser nominal, no apto para
un índice continuo). Rango tentativo: `(0.2, 0.7)` (`x5_contexto_conversacion.md`
§8 — **no está en `X5_PARAM_RANGES`** todavía, ver `plan_x5_alt_v2.md`
sección 6). `k=2` → el ajuste explora el peso relativo de ambas variables
igual que en la Sección 4.

### 6.3. Qué se reportará

Mismo mecanismo de la Sección 4 (índice compuesto de 2 variables →
suavizado → percentil causal → sigmoide), sin reexplicarlo acá. Mismos
entregables: gráfico del índice compuesto, curva ajustada, serie dinámica
vs. estática, diagnóstico, export a `{valor}_q4_factor_b.csv`.

## 7. K = f(persistencia de niveles) — ¿cuánto pesa que un nivel se haya mantenido intacto?

**Pasos**: `[Q4-13]` implementar → `[Q4-14]` ejecutar → `[Q4-15]` revisar
visuales y validar antes de pasar a `N_EXP`.

### 7.0. Qué controla `K` en producción

En X0, `K` pondera el aislamiento temporal futuro de una vela respecto del
pasado en el scoring de soportes: `y = Low_left + High_left + K *
(Low_right + High_right)` (`x5_contexto_conversacion.md` §12). `K` alto
prioriza niveles que permanecieron intactos mucho tiempo después de
formarse (validados por el tiempo transcurrido); `K` bajo prioriza el
contexto que originó el nivel antes de formarse, por sobre su validación
posterior.

### 7.1. Hipótesis vigente

> mercado estable y niveles persistentes → `K` más alto. muchos cambios de
> régimen o rupturas → `K` más bajo.

**Nota importante**: `K` es **global en producción**, no por activo
(`pars_configurables.md`). Esta sección lo calibra por activo igual que el
resto, por consistencia con `Q1`-`Q3` — pero llevarlo a producción exigiría
antes decidir si `K` pasa a ser por-activo también en `X0`, decisión
arquitectónica fuera de esta fase (`plan_x5_alt_v2.md` §6).

### 7.2. Variable elegida y por qué

*Pendiente de implementar.* Variable: `ruptura_rango` (se descarta
`duracion_en_rango`, evidencia ≈0 en `Q3-4`). Rango: `(0.5, 2.0)`
(`X5_PARAM_RANGES['K']`), `k=1`.

### 7.3. Qué se reportará

Mismo mecanismo de la Sección 5 (`k=1`, sin ponderación de variables), sin
reexplicarlo acá. Export a `{valor}_q4_k.csv`.

## 8. N_EXP = f(cambio de volatilidad) — ¿cuánto debe pesar lo reciente al ponderar velas?

**Pasos**: `[Q4-16]` implementar → `[Q4-17]` ejecutar → `[Q4-18]` revisar
visuales y validar antes de pasar al resumen consolidado.

### 8.0. Qué controla `N_EXP` en producción

En X0, `N_EXP` es el exponente de recencia del peso de cada vela: `w = t **
N_EXP`, con `t∈[0,1]` (0=vela más antigua del período, 1=la más reciente)
(`x5_contexto_conversacion.md` §13). `N_EXP` alto concentra el peso en lo
reciente (las velas nuevas dominan la función objetivo de soportes);
`N_EXP` bajo reparte el peso de forma más pareja en todo el historial (más
memoria, menos reactivo a cambios recientes).

### 8.1. Hipótesis vigente

> mercado estable y repetitivo → `N_EXP` más bajo (conservar memoria
> histórica, no sobre-reaccionar a lo último). cambio de régimen o
> volatilidad creciente → `N_EXP` más alto (lo reciente importa más cuando
> el contexto cambió).

**Nota**: igual que `K`, `N_EXP` es global en producción, no por activo —
se calibra por activo acá por consistencia con el resto de la fase, pero
llevarlo a producción exige la misma decisión arquitectónica pendiente que
`K` (`plan_x5_alt_v2.md` §6).

### 8.2. Variable elegida y por qué

*Pendiente de implementar.* Variable: `cambio_volatilidad` (ratio
volatilidad corta/larga — se descarta `frecuencia_extremos_30d` para
mantener `k=1`, era secundaria según `Q3-6`). Rango: `(0.5, 3.0)`
(`X5_PARAM_RANGES['N_EXP']`), `k=1`.

### 8.3. Qué se reportará

Mismo mecanismo de las Secciones 5/7 (`k=1`), sin reexplicarlo acá. Export a
`{valor}_q4_n_exp.csv`.

## 9. Resumen consolidado (de los 5 parámetros: N, A, factor_B, K, N_EXP)

**Pasos**: `[Q4-19]` implementar → `[Q4-20]` ejecutar y revisar la tabla
completa.

*Pendiente.* Tabla única por activo con bondad de ajuste (R²), elasticidad
en el punto medio y señal de coincidencia de signo vs. la hipótesis de `Q3`,
para los 5 combos calibrados. Export a `{valor}_q4_resumen.csv`.

## 10. Export a zip (Q4-EXEC)

Las secciones 4-9 escriben sus CSV en `CARPETA_EXP`
(`resources/x5_alt/exp_Q4/`, limpiada en la Sección 0 por `limpiar_exp`) —
mismo patrón ya aplicado en `X5_Q1`/`X5_Q2`/`X5_Q3.ipynb`. Esta celda final
solo llama `exportar_zip('X5_Q4')` (`scripts/x5_alt_export.py`) para
comprimir `exp_Q4` completa en `exp_Q4.zip` — corre sobre lo que exista en
ese momento en `CARPETA_EXP`, sin exigir que las Secciones 8-9 (N_EXP,
resumen consolidado) ya estén implementadas; se puede volver a correr esta
celda cuantas veces haga falta a medida que el notebook avanza, para mover
`exp_Q4/` de Windows a Mac en cualquier punto intermedio. Última corrida de
punta a punta (6 activos) — cierre formal de la Fase 2 principal (sin
`LAMBDA`, ver Sección 11).

In [ ]:
exportar_zip('X5_Q4')

## 11. Extensión opcional — LAMBDA (fuera del alcance comprometido de esta fase)

Mauricio decidió dejar `LAMBDA` explícitamente fuera del alcance de esta
primera iteración (`plan_x5_alt_v2.md` sección 0). Retomar esta sección solo
si decide resolver su bloqueador — no es parte del cierre de `[Q4-EXEC]`.

**Pasos**: `[Q4-L1]` resolver y documentar el rango real de `LAMBDA` (el
rango de exploración documentado, `0.001`-`0.02`, no coincide con el valor
productivo `0.2`, `config.py:95` — bloqueador heredado de `Q3-6`) →
`[Q4-L2]` implementar `LAMBDA = f(x3_bb_width, duracion_en_rango)` (`k=2`,
mismo esqueleto de las secciones 4-8) → `[Q4-L3]` ejecutar `X5_Q4.ipynb`
completo → `[Q4-L4]` revisar visuales y validar.